## first try to run the cresi pipeline, to see how it works

In [19]:
import os
import cv2
import numpy as np
import subprocess
import matplotlib.pyplot as plt

In [27]:
BASE_DIR = os.path.abspath() # project root
INPUT_DIR = os.path.join(BASE_DIR, "data/1_input_raw")
MASK_DIR = os.path.join(BASE_DIR, "data/2_cloud_masks")
CLEAN_DIR = os.path.join(BASE_DIR, "data/3_inpainted_clean")
CRESI_OUT = os.path.join(BASE_DIR, "data/4_cresi_output")
print(BASE_DIR)

D:\Leiden Universiteit\urban computing\project


In [21]:
def run_cresi_extraction(config_file="sn5_baseline.json"):
    cmd = [
        "docker", "run", "--gpus", "all", "--rm",
        "-v", f"{BASE_DIR}/cresi:/cresi",
        "-v", f"{CLEAN_DIR}:/cresi/data/test_images", # Map clean images to test folder
        "-v", f"{CRESI_OUT}:/cresi/results",
        "cresi_image",                                # The image we built earlier
        "python", "/cresi/cresi/02_eval.py", config_file
    ]

    print("Starting CRESI Extraction...")
    subprocess.run(cmd)
    print("CRESI Finished.")

In [29]:
# Create a dummy file in the input folder
!echo "test file" > {CLEAN_DIR}/test_file.txt

# Run a test container to list the contents of the mounted folder
cmd = [
    "docker", "run", "--rm",
    "-v", f"{CLEAN_DIR}:/data",
    "alpine",
    "ls", "-al", "/data"
]
subprocess.run(cmd)


CompletedProcess(args=['docker', 'run', '--rm', '-v', 'D:\\Leiden Universiteit\\urban computing\\project\\data/3_inpainted_clean:/data', 'alpine', 'ls', '-al', '/data'], returncode=0)

In [22]:
def run_merge_preds(config_file="sn5_baseline.json"):
    cmd = [
        "docker", "run", "--gpus", "all", "--rm",
        "-v", f"{BASE_DIR}/cresi:/cresi",
        "-v", f"{CRESI_OUT}:/cresi/results",
        "cresi_image",
        "python", "/cresi/cresi/03a_merge_preds.py", config_file
    ]
    print("Merging Predictions...")
    subprocess.run(cmd)

def run_stitch(config_file="sn5_baseline.json"):
    cmd = [
        "docker", "run", "--gpus", "all", "--rm",
        "-v", f"{BASE_DIR}/cresi:/cresi",
        "-v", f"{CRESI_OUT}:/cresi/results",
        "cresi_image",
        "python", "/cresi/cresi/03b_stitch.py", config_file
    ]
    print("Stitching Mask Windows...")
    subprocess.run(cmd)

In [23]:
def run_skeletonize(config_file="sn5_baseline.json"):
    cmd = [
        "docker", "run", "--gpus", "all", "--rm",
        "-v", f"{BASE_DIR}/cresi:/cresi",
        "-v", f"{CRESI_OUT}:/cresi/results",
        "cresi_image",
        "python", "/cresi/cresi/04_skeletonize.py", config_file
    ]
    print("Extracting Mask Skeletons...")
    subprocess.run(cmd)

In [24]:
def run_create_graph(config_file="sn5_baseline.json"):
    cmd = [
        "docker", "run", "--gpus", "all", "--rm",
        "-v", f"{BASE_DIR}/cresi:/cresi",
        "-v", f"{CRESI_OUT}:/cresi/results",
        "cresi_image",
        "python", "/cresi/cresi/05_wkt_to_G.py", config_file
    ]
    print("Creating Road Graph (NetworkX)...")
    subprocess.run(cmd)

In [25]:
def run_infer_speed(config_file="sn5_baseline.json"):
    cmd = [
        "docker", "run", "--gpus", "all", "--rm",
        "-v", f"{BASE_DIR}/cresi:/cresi",
        "-v", f"{CRESI_OUT}:/cresi/results",
        "cresi_image",
        "python", "/cresi/cresi/06_infer_speed.py", config_file
    ]
    print("Inferring Road Speed and Travel Time...")
    subprocess.run(cmd)

In [26]:
config_file = "sn5_baseline.json" # Or whatever your config is
run_cresi_extraction(config_file)
run_skeletonize(config_file)
run_create_graph(config_file)
run_infer_speed(config_file)

print("\n🎉 Pipeline Complete! Check your CRESI_OUT folder for WKT, Graph files, and speed overlays!")

Starting CRESI Extraction...
CRESI Finished.
Extracting Mask Skeletons...
Creating Road Graph (NetworkX)...
Inferring Road Speed and Travel Time...

🎉 Pipeline Complete! Check your CRESI_OUT folder for WKT, Graph files, and speed overlays!
